# Artificial Neural Networks — Sonar: Mines vs. Rocks

## Business Objective

##### To build an intelligent system that automatically detects whether an underwater sonar signal is reflected from a metallic mine (dangerous) or a harmless rock.
##### This is vital for:
##### -Maritime safety: Prevent ships/submarines from colliding with mines.
##### -Naval defense: Identify and safely remove underwater mines.
##### -Resource exploration: Distinguish metal structures from natural seabed objects.

## Problem Statement
##### Sonar signals in underwater environments are noisy and difficult for humans to interpret consistently. This dataset contains 208 sonar returns:
#####  -111 from metal cylinders (Mines) — labelled M
##### -97 from rocks — labelled R
##### -Each return has 60 numeric features representing signal energy per frequency band.
##### Goal: Train a Deep Learning (ANN) model to classify new sonar signals as Mine (M) or Rock (R) — accurately and reliably.

In [1]:
import sys
print(sys.version)

3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]


In [2]:
!pip --version

pip 25.3 from C:\Users\h238p\anaconda3\Lib\site-packages\pip (python 3.13)



In [3]:
!conda install scikit-learn -y

Jupyter detected...
3 channel Terms of Service accepted
Channels:
 - defaults
Platform: win-64
Solving environment: done

# All requested packages already installed.





==> WARNING: A newer version of conda exists. <==
    current version: 26.1.1
    latest version: 26.5.3

Please update conda by running

    $ conda update -n base -c defaults conda




In [4]:
!pip install -q scikit-learn==1.4.2 scikeras==0.13.0 tensorflow

  error: subprocess-exited-with-error
  
  installing build dependencies for scikit-learn did not run successfully.
  exit code: 1
  
  [10 lines of output]
    Using cached setuptools-83.0.0-py3-none-any.whl.metadata (6.6 kB)
    Using cached wheel-0.47.0-py3-none-any.whl.metadata (2.3 kB)
    Using cached cython-3.2.9-cp313-cp313-win_amd64.whl.metadata (4.7 kB)
  ERROR: Ignored the following yanked versions: 2.4.0
  ERROR: Ignored the following versions that require a different python version: 1.21.2 Requires-Python >=3.7,<3.11; 1.21.3 Requires-Python >=3.7,<3.11; 1.21.4 Requires-Python >=3.7,<3.11; 1.21.5 Requires-Python >=3.7,<3.11; 1.21.6 Requires-Python >=3.7,<3.11; 1.26.0 Requires-Python >=3.9,<3.13; 1.26.1 Requires-Python >=3.9,<3.13
  ERROR: Could not find a version that satisfies the requirement numpy==2.0.0rc1 (from versions: 1.3.0, 1.4.1, 1.5.0, 1.5.1, 1.6.0, 1.6.1, 1.6.2, 1.7.0, 1.7.1, 1.7.2, 1.8.0, 1.8.1, 1.8.2, 1.9.0, 1.9.1, 1.9.2, 1.9.3, 1.10.0.post2, 1.10.1, 1.10.2, 1.

In [5]:
#importing libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
import warnings
warnings.filterwarnings('ignore')
print("TensorFlow version:", tf.__version__)

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
import scikeras
import sklearn

print(f"Scikeras version: {scikeras.__version__}")
print(f"Scikit-learn version: {sklearn.__version__}")

In [ ]:
df = pd.read_csv("sonardataset.csv")

df.head()

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
#Check Target Classes
df.iloc[:,-1].value_counts()

In [ ]:
#Separate Features and Target
X = df.iloc[:,:-1]

y = df.iloc[:,-1]

In [ ]:
#Encode Target Variable
encoder = LabelEncoder()
y = encoder.fit_transform(y)

In [ ]:
#Normalize the Data
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [ ]:
#Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)
print(X_train.shape)
print(X_test.shape)

print(y_train.shape)
print(y_test.shape)

In [ ]:
#Class Distribution Plot
plt.figure(figsize=(5, 4))
df['Y'].value_counts().plot(kind='bar', color=['steelblue', 'coral'], edgecolor='black')
plt.title('Class Distribution')
plt.xlabel('Class')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
#Correlation Heatmap
plt.figure(figsize=(10, 6))
sns.heatmap(df.iloc[:, :10].corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Heatmap (First 10 Features)')
plt.tight_layout()
plt.show()

In [ ]:
#preprocessing
X = df.drop('Y', axis=1).values
y = df['Y'].values

le = LabelEncoder()
y = le.fit_transform(y)
print('Label encoding:', dict(zip(le.classes_, le.transform(le.classes_))))

scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'\nTrain: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
#Build Baseline ANN
def build_baseline_model():
    model = keras.Sequential([
        layers.Input(shape=(60,)),
        layers.Dense(64, activation='relu'),
        layers.Dense(32, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

baseline_model = build_baseline_model()
baseline_model.summary()
     

In [ ]:
#Train Baseline Model
early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

history_baseline = baseline_model.fit(
    X_train, y_train,
    epochs=200,
    batch_size=16,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
#Plot Baseline Training History
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history_baseline.history['accuracy'], label='Train Accuracy')
axes[0].plot(history_baseline.history['val_accuracy'], label='Val Accuracy')
axes[0].set_title('Baseline Model — Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(history_baseline.history['loss'], label='Train Loss')
axes[1].plot(history_baseline.history['val_loss'], label='Val Loss')
axes[1].set_title('Baseline Model — Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
#Evaluate Baseline
y_pred_baseline = (baseline_model.predict(X_test) > 0.5).astype(int).flatten()

print('=== Baseline Model Evaluation ===')
print(f'Accuracy  : {accuracy_score(y_test, y_pred_baseline):.4f}')
print(f'Precision : {precision_score(y_test, y_pred_baseline):.4f}')
print(f'Recall    : {recall_score(y_test, y_pred_baseline):.4f}')
print(f'F1-Score  : {f1_score(y_test, y_pred_baseline):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_baseline, target_names=le.classes_))

cm = confusion_matrix(y_test, y_pred_baseline)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Baseline — Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

In [ ]:
#Hyperparameter Tuning Setup
def build_tunable_model(neurons=64, hidden_layers=2, activation='relu', learning_rate=0.001):
    model = keras.Sequential()
    model.add(layers.Input(shape=(60,)))
    for _ in range(hidden_layers):
        model.add(layers.Dense(neurons, activation=activation))
    model.add(layers.Dense(1, activation='sigmoid'))
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

keras_clf = KerasClassifier(model=build_tunable_model, epochs=100, batch_size=16, verbose=0)

param_grid = {
    'model__neurons'       : [32, 64, 128],
    'model__hidden_layers' : [1, 2, 3],
    'model__activation'    : ['relu', 'tanh'],
    'model__learning_rate' : [0.001, 0.01]
}

In [ ]:
#RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=keras_clf,
    param_distributions=param_grid,
    n_iter=10,
    cv=3,
    scoring='accuracy',
    random_state=42,
    n_jobs=1
)

random_search.fit(X_train, y_train)

print('\nBest Parameters:', random_search.best_params_)
print(f'Best CV Accuracy: {random_search.best_score_:.4f}')

cv_results = pd.DataFrame(random_search.cv_results_)
cols = ['param_model__neurons', 'param_model__hidden_layers',
        'param_model__activation', 'param_model__learning_rate',
        'mean_test_score', 'std_test_score', 'rank_test_score']
print(cv_results[cols].sort_values('rank_test_score').to_string(index=False))

In [ ]:
#Train Tuned Model
best_params = random_search.best_params_

tuned_model = build_tunable_model(
    neurons       = best_params['model__neurons'],
    hidden_layers = best_params['model__hidden_layers'],
    activation    = best_params['model__activation'],
    learning_rate = best_params['model__learning_rate']
)
tuned_model.summary()

early_stop2 = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)

history_tuned = tuned_model.fit(
    X_train, y_train,
    epochs=200,
    batch_size=16,
    validation_split=0.2,
    callbacks=[early_stop2],
    verbose=1
)

In [ ]:
#Evaluate Tuned Model
y_pred_tuned = (tuned_model.predict(X_test) > 0.5).astype(int).flatten()

print('=== Tuned Model Evaluation ===')
print(f'Accuracy  : {accuracy_score(y_test, y_pred_tuned):.4f}')
print(f'Precision : {precision_score(y_test, y_pred_tuned):.4f}')
print(f'Recall    : {recall_score(y_test, y_pred_tuned):.4f}')
print(f'F1-Score  : {f1_score(y_test, y_pred_tuned):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_tuned, target_names=le.classes_))

cm2 = confusion_matrix(y_test, y_pred_tuned)
plt.figure(figsize=(5, 4))
sns.heatmap(cm2, annot=True, fmt='d', cmap='Greens',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Tuned Model — Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

In [ ]:
#Baseline vs Tuned Comparison
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
baseline_scores = [
    accuracy_score(y_test, y_pred_baseline),
    precision_score(y_test, y_pred_baseline),
    recall_score(y_test, y_pred_baseline),
    f1_score(y_test, y_pred_baseline)
]
tuned_scores = [
    accuracy_score(y_test, y_pred_tuned),
    precision_score(y_test, y_pred_tuned),
    recall_score(y_test, y_pred_tuned),
    f1_score(y_test, y_pred_tuned)
]

comparison_df = pd.DataFrame({
    'Metric'        : metrics,
    'Baseline Model': baseline_scores,
    'Tuned Model'   : tuned_scores
})
comparison_df['Improvement'] = comparison_df['Tuned Model'] - comparison_df['Baseline Model']
print(comparison_df.to_string(index=False))

x = np.arange(len(metrics))
width = 0.35
fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, baseline_scores, width, label='Baseline', color='steelblue')
bars2 = ax.bar(x + width/2, tuned_scores,    width, label='Tuned',    color='coral')
ax.set_xlabel('Metric')
ax.set_ylabel('Score')
ax.set_title('Baseline vs Tuned Model — Performance Comparison')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.1)
ax.legend()
for bar in bars1 + bars2:
    ax.annotate(f'{bar.get_height():.3f}',
                xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 3), textcoords='offset points', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

#### DISCUSSION: Baseline vs Tuned Model
#### BASELINE MODEL:

##### Architecture : Input → Dense(64, ReLU) → Dense(32, ReLU) → Output
##### Accuracy : 0.8810
##### Precision : 0.9412
##### Recall : 0.8000
##### F1-Score : 0.8649

#### TUNED MODEL (Best from RandomizedSearchCV):

##### Architecture : Input → Dense(128, ReLU) × 2 → Output
##### Accuracy : 0.8810
##### Precision : 1.0000
##### Recall : 0.7500
##### F1-Score : 0.8571

#### HYPERPARAMETER TUNING METHODOLOGY:

##### Method : RandomizedSearchCV (n_iter=10, cv=3)
##### Search Space : neurons → [32, 64, 128] hidden_layers → [1, 2, 3] activation → ['relu', 'tanh'] learning_rate → [0.001, 0.01]
##### Best Config : 128 neurons, 2 layers, ReLU, lr=0.001
##### Best CV Acc : 0.8317
##### EFFECT OF HYPERPARAMETER TUNING:

##### Overall accuracy remained the same (88.1%), showing the baseline was already well-configured for this dataset size.
##### Precision improved significantly: from 0.9412 → 1.0000. The tuned model never incorrectly predicted a Mine — zero false positives. This is critical in real-world naval defense where a false Mine alert is costly but a missed Mine is catastrophic.
##### Recall decreased slightly: from 0.8000 → 0.7500. The tuned model is more conservative — it only predicts Mine when very confident, missing some true Mines (higher false negatives).
###### F1-Score stayed similar (0.8649 vs 0.8571), confirming the overall balance between precision and recall is comparable.

##### CONCLUSION: Hyperparameter tuning with RandomizedSearchCV helped identify a model that is more precise and reliable for Mine detection. Depending on the use case — if zero false positives is the priority (naval safety), the tuned model is the better choice.